# Augmenting the dataset

In [1]:
import numpy as np
from scipy.ndimage import shift
import os

In [2]:
DATASET_PATH   = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\processed\dataset.npz"
OUTPUT_PATH    = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\processed\dataset_augmented.npz"

In [3]:
IMAGE_SIZE  = (224, 224)  # Must match what you used in preprocess.py
CHANNELS    = 3           # RGB
SHIFT_PX    = 3           # How many pixels to shift in each direction

In [4]:
# 4 shift directions (row_shift, col_shift)
SHIFT_DIRECTIONS = [
    (-SHIFT_PX, 0),  # Up
    (SHIFT_PX,  0),  # Down
    (0, -SHIFT_PX),  # Left
    (0,  SHIFT_PX),  # Right
]

In [5]:
def shift_image(flat_img, row_shift, col_shift):
    """
    Takes a flat 1D image array, reshapes it to 3D (H x W x C),
    shifts it by the given number of pixels, then flattens back to 1D.
    Pixels shifted out of frame are filled with 0 (black).
    """
    img = flat_img.reshape(IMAGE_SIZE[0], IMAGE_SIZE[1], CHANNELS)
 
    # Shift each channel independently
    shifted = np.zeros_like(img)
    for c in range(CHANNELS):
        shifted[:, :, c] = shift(
            img[:, :, c],
            shift=[row_shift, col_shift],
            mode='constant',
            cval=0.0
        )
 
    return shifted.flatten()

In [6]:
def augment_dataset(X, y):
    """
    Takes X (images) and y (labels) and returns a new augmented
    dataset containing the originals plus all 8 shifted versions.
    """
    total         = len(X)
    num_augmented = total * len(SHIFT_DIRECTIONS)
 
    print(f"  Original images:   {total}")
    print(f"  Augmented images:  {num_augmented}")
    print(f"  Total after aug:   {total + num_augmented}")
    print(f"  (9x original size)\n")
 
    X_augmented = np.zeros((num_augmented, X.shape[1]), dtype=X.dtype)
    y_augmented = np.zeros(num_augmented, dtype=y.dtype)
 
    idx = 0
    for i, (img, label) in enumerate(zip(X, y)):
        if i % 500 == 0:
            print(f"  Augmenting image {i+1}/{total}...")
 
        for row_shift, col_shift in SHIFT_DIRECTIONS:
            X_augmented[idx] = shift_image(img, row_shift, col_shift)
            y_augmented[idx] = label
            idx += 1
 
    # Combine originals with augmented
    X_combined = np.concatenate([X, X_augmented])
    y_combined = np.concatenate([y, y_augmented])
 
    return X_combined, y_combined

In [7]:
def print_class_distribution(y, label=""):
    """Print how many images exist per grade."""
    print(f"\nClass distribution {label}:")
    print(f"{'Grade':<10} {'Count':<10}")
    print(f"{'-'*20}")
    for grade in range(1, 11):
        count = np.sum(y == grade)
        print(f"PSA {grade:<6} {count:<10}")
    print(f"{'Total':<10} {len(y):<10}")

In [8]:
def main():
    print("Loading dataset...")
    data = np.load(DATASET_PATH)

    X_train = data['X_train']
    X_val   = data['X_val']
    X_test  = data['X_test']
    y_train = data['y_train']
    y_val   = data['y_val']
    y_test  = data['y_test']

    print(f"  Training samples:   {len(X_train)}")
    print_class_distribution(y_train, "(before augmentation)")

    # Augment in chunks to avoid running out of memory
    chunk_size = 500
    X_aug_list = [X_train]
    y_aug_list = [y_train]

    for direction in SHIFT_DIRECTIONS:
        print(f"  Applying shift {direction}...")
        X_shifted = np.zeros_like(X_train)
        for i in range(0, len(X_train), chunk_size):
            chunk = X_train[i:i+chunk_size]
            for j, img in enumerate(chunk):
                X_shifted[i+j] = shift_image(img, direction[0], direction[1])
        X_aug_list.append(X_shifted)
        y_aug_list.append(y_train)

    X_train_aug = np.concatenate(X_aug_list)
    y_train_aug = np.concatenate(y_aug_list)

    print_class_distribution(y_train_aug, "(after augmentation)")

    print(f"\nSaving augmented dataset...")
    np.savez_compressed(
        OUTPUT_PATH,
        X_train=X_train_aug,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train_aug,
        y_val=y_val,
        y_test=y_test
    )

    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"Saved to: {OUTPUT_PATH}")
    print(f"File size: {size_mb:.1f} MB")
    print(f"\nDone! Update DATASET_PATH in your training scripts to point to:")
    print(f"  {OUTPUT_PATH}")

In [9]:
main()

Loading dataset...
  Training samples:   5342

Class distribution (before augmentation):
Grade      Count     
--------------------
PSA 1      34        
PSA 2      20        
PSA 3      262       
PSA 4      287       
PSA 5      610       
PSA 6      339       
PSA 7      1230      
PSA 8      1326      
PSA 9      1016      
PSA 10     218       
Total      5342      
  Applying shift (-3, 0)...
  Applying shift (3, 0)...
  Applying shift (0, -3)...
  Applying shift (0, 3)...

Class distribution (after augmentation):
Grade      Count     
--------------------
PSA 1      170       
PSA 2      100       
PSA 3      1310      
PSA 4      1435      
PSA 5      3050      
PSA 6      1695      
PSA 7      6150      
PSA 8      6630      
PSA 9      5080      
PSA 10     1090      
Total      26710     

Saving augmented dataset...
Saved to: C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\processed\dataset_augmented.npz
File size: 5223.7 MB

Done! Update DATA